In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
import os

# Change this to your actual project root path
project_root = "E:/HealthyBites"
os.chdir(project_root)

print(f"Now working in: {os.getcwd()}")

Now working in: E:\HealthyBites


In [3]:
# Indian Dataset
df_indian = pd.read_csv("data/raw/Cleaned_Indian_Food_Dataset.csv")

# Original Dataset (Food.com)
df_original = pd.read_csv("data/processed/clean_recipes.csv")

print("Indian dataset shape:", df_indian.shape)
print("Original dataset shape:", df_original.shape)


Indian dataset shape: (5938, 9)
Original dataset shape: (230186, 6)


In [4]:
print("Indian Columns:\n", df_indian.columns)
print("\nOriginal Columns:\n", df_original.columns)


Indian Columns:
 Index(['TranslatedRecipeName', 'TranslatedIngredients', 'TotalTimeInMins',
       'Cuisine', 'TranslatedInstructions', 'URL', 'Cleaned-Ingredients',
       'image-url', 'Ingredient-count'],
      dtype='str')

Original Columns:
 Index(['id', 'name', 'minutes', 'ingredients', 'steps', 'calories'], dtype='str')


In [5]:
import numpy as np
import ast

# Remove missing important fields
df_original = df_original.dropna(subset=["ingredients", "steps", "calories"])

# Convert ingredients from string to list (if needed)
def parse_ingredients(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df_original["ingredients"] = df_original["ingredients"].apply(parse_ingredients)

# Keep recipes with at least 4 ingredients
df_original = df_original[df_original["ingredients"].map(len) >= 4]

# Keep recipes with cooking time > 5 minutes
df_original = df_original[df_original["minutes"] > 5]

print("Filtered original shape:", df_original.shape)


Filtered original shape: (209549, 6)


In [6]:
df_original_sample = df_original.sample(n=200000, random_state=42)

print("Sampled original shape:", df_original_sample.shape)


Sampled original shape: (200000, 6)


In [7]:
df_indian = df_indian.rename(columns={
    "TranslatedRecipeName": "name",
    "TranslatedIngredients": "ingredients",
    "TranslatedInstructions": "instructions",
    "TotalTimeInMins": "minutes"

})

df_original_sample = df_original_sample.rename(columns={
    "steps": "instructions"
})



In [8]:
df_indian["calories"] = np.nan
df_indian["source"] = "indian"

df_original_sample["source"] = "original"


In [9]:
# If Cuisine not present in original, create it
df_original_sample["Cuisine"] = "unknown"

df_indian = df_indian[
    ["name", "ingredients", "instructions", "minutes", "calories", "Cuisine", "source"]
]

df_original_sample = df_original_sample[
    ["name", "ingredients", "instructions", "minutes", "calories", "Cuisine", "source"]
]


In [10]:
def clean_ingredients(lst):
    try:
        return [str(i).strip().lower() for i in lst]
    except:
        return []

df_indian["ingredients"] = df_indian["ingredients"].apply(parse_ingredients)
df_indian["ingredients"] = df_indian["ingredients"].apply(clean_ingredients)

df_original_sample["ingredients"] = df_original_sample["ingredients"].apply(clean_ingredients)


In [11]:
def clean_text(x):
    x = str(x).lower().strip()
    x = x.replace("\n", " ").replace("\r", " ")
    return x

for df in [df_indian, df_original_sample]:
    df["name"] = df["name"].apply(clean_text)
    df["instructions"] = df["instructions"].apply(clean_text)


In [12]:
df_combined = pd.concat([df_indian, df_original_sample], ignore_index=True)

print("Combined shape:", df_combined.shape)


Combined shape: (205938, 7)


In [13]:
df_combined = df_combined[
    df_combined["ingredients"].map(len) > 0
]

df_combined = df_combined.reset_index(drop=True)

print("After removing empty ingredients:", df_combined.shape)


After removing empty ingredients: (200000, 7)


In [14]:
# Convert ingredients to tuple for deduplication
df_combined["ingredient_tuple"] = df_combined["ingredients"].apply(lambda x: tuple(sorted(x)))

# Drop duplicates based on name + ingredient_tuple
df_combined = df_combined.drop_duplicates(
    subset=["name", "ingredient_tuple"]
)

# Remove helper column
df_combined = df_combined.drop(columns=["ingredient_tuple"])

print("After removing duplicates:", df_combined.shape)


After removing duplicates: (200000, 7)


In [15]:
df_combined.info()
df_combined.head()


<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   name          200000 non-null  str    
 1   ingredients   200000 non-null  object 
 2   instructions  200000 non-null  str    
 3   minutes       200000 non-null  int64  
 4   calories      200000 non-null  float64
 5   Cuisine       200000 non-null  str    
 6   source        200000 non-null  str    
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 10.7+ MB


,name,ingredients,instructions,minutes,calories,Cuisine,source
0,best chicken n dumplings,"[chicken broth, boneless skinless chicken thig...",['in large soup pot bring chicken broth to a b...,50,264.3,unknown,original
1,el chaya shrimp enchiladas,"[olive oil, diced onion, roma tomato, shrimp, ...",['heat olive oil in skillet over medium-high h...,50,520.0,unknown,original
2,goulash canning,"[chuck roast, salt, paprika, dry mustard, oil,...","['cut the beef chuck roast into 1"" cubes', 'co...",190,2839.2,unknown,original
3,fiery indian fruit salad,"[fresh fruit, fresh lemon juice, cayenne, cumi...","['place fruit in a large serving bowl', 'mix i...",105,11.5,unknown,original
4,candy bar mousse pie,"[milk chocolate candy bars with almonds, marsh...","['place one candy bar , marshmallows and milk ...",10,869.0,unknown,original


In [17]:
df_combined.to_csv("data/processed/healthybites_master_dataset.csv", index=False)

print("✅ Master dataset saved successfully.")


✅ Master dataset saved successfully.


In [19]:
# Step 1: Import pandas library
import pandas as pd

# Step 2: Load the nutrition dataset (change the path if needed)
df_nutrition = pd.read_csv('data/raw/calories.csv')

# Step 3: Check the first few rows to understand its structure
df_nutrition.head()


,FoodCategory,FoodItem,per100grams,Cals_per100grams,KJ_per100grams
0,CannedFruit,Applesauce,100g,62 cal,260 kJ
1,CannedFruit,Canned Apricots,100g,48 cal,202 kJ
2,CannedFruit,Canned Blackberries,100g,92 cal,386 kJ
3,CannedFruit,Canned Blueberries,100g,88 cal,370 kJ
4,CannedFruit,Canned Cherries,100g,54 cal,227 kJ


In [20]:
%who


ast	 clean_ingredients	 clean_text	 df	 df_combined	 df_indian	 df_nutrition	 df_original	 df_original_sample	 
np	 os	 parse_ingredients	 pd	 project_root	 


In [21]:
# Step 2: Keep only necessary columns ('FoodItem' and 'Cals_per100grams')
df_nutrition = df_nutrition[['FoodItem', 'Cals_per100grams']]

# Step 3: Rename columns for consistency
df_nutrition = df_nutrition.rename(columns={'FoodItem': 'ingredient', 'Cals_per100grams': 'calories_per_100g'})

# Step 4: Clean and normalize the ingredient names
df_nutrition['ingredient'] = df_nutrition['ingredient'].str.lower().str.strip()

# Step 5: Check the cleaned dataset
df_nutrition.head()


,ingredient,calories_per_100g
0,applesauce,62 cal
1,canned apricots,48 cal
2,canned blackberries,92 cal
3,canned blueberries,88 cal
4,canned cherries,54 cal


In [22]:
# Check the actual column names in the dataset
df_nutrition.columns


Index(['ingredient', 'calories_per_100g'], dtype='str')

In [63]:
# Step 3: Create a calorie lookup dictionary
ingredient_cal_dict = dict(zip(df_nutrition['ingredient'], df_nutrition['calories_per_100g']))

# Check the dictionary by printing a few entries
for ingredient, calories in list(ingredient_cal_dict.items())[:10]:  # Displaying first 10 entries
    print(f"{ingredient}: {calories}")


applesauce: 68 cal
canned apricots: 48 cal
canned blackberries: 92 cal
canned blueberries: 88 cal
canned cherries: 54 cal
canned cranberries: 178 cal
canned crushed pineapple: 53 cal
canned figs: 107 cal
canned fruit cocktail: 81 cal
canned fruit salad: 50 cal


In [64]:
# Function to convert calories (remove ' cal' and convert to numeric)
def convert_to_numeric(calories_str):
    try:
        return float(calories_str.replace(' cal', '').strip())  # Remove " cal" and convert to float
    except ValueError:
        return 0  # Return 0 if the conversion fails

# Function to calculate calories for a recipe based on its ingredients
def estimate_calories_with_core_ingredient(ingredients):
    total_calories = 0
    for ingredient in ingredients:
        # Extract the core ingredient
        core_ingredient = extract_core_ingredient(ingredient)
        
        # Look up calories using the core ingredient
        calories_str = ingredient_cal_dict.get(core_ingredient, "0 cal")  # Default to "0 cal" if not found
        calories = convert_to_numeric(calories_str)  # Convert string to numeric
        total_calories += calories
    return total_calories

# Example: Applying the function to the 'ingredients' column
df_combined['calories'] = df_combined['ingredients'].apply(estimate_calories_with_core_ingredient)

# Check the first few rows after adding calories
df_combined[['name', 'ingredients', 'calories']].head()


,name,ingredients,calories
0,best chicken n dumplings,"[chicken broth, boneless skinless chicken thig...",283.0
1,el chaya shrimp enchiladas,"[olive oil, diced onion, roma tomato, shrimp, ...",1525.0
2,goulash canning,"[chuck roast, salt, paprika, dry mustard, oil,...",790.0
3,fiery indian fruit salad,"[fresh fruit, fresh lemon juice, cayenne, cumi...",678.0
4,candy bar mousse pie,"[milk chocolate candy bars with almonds, marsh...",379.0


In [65]:
# Check for any missing values in the 'calories' column
print(df_combined['calories'].isnull().sum())


0


In [66]:
# Fill missing calories with a default value or average calories per cuisine
df_combined['calories'] = df_combined['calories'].fillna(df_combined['calories'].mean())


In [67]:
# Save the updated dataset with calories
df_combined.to_csv('data/processed/healthybites_master_dataset_full_calories.csv', index=False)


In [70]:
import pandas as pd
import ast

# Read the dataset from the CSV file
df = pd.read_csv("data/processed/healthybites_master_dataset_full_calories.csv")

# Convert the ingredients column from string to Python list (if necessary)
df["ingredients"] = df["ingredients"].apply(ast.literal_eval)

# Extract the ingredients column
ingredients_column = df[["ingredients"]]

# Save the ingredients column to a new CSV file
ingredients_column.to_csv("data/sampled/ingredients_only.csv", index=False)

print("Ingredients column has been saved to data/sampled/ingredients_only.csv")


Ingredients column has been saved to data/sampled/ingredients_only.csv
